离线构造教学语言视图。核心代码、已有输出与配置沿已接受读者源；固定申报事实/图属于历史教学输入。下列新计算不构成市场策略复现。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch13_valuation_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第13章：面向量化研究的估值

Notebook 直接导入 src/quant_course/valuation.py。Orion、Lyra、估值数值、PIT 时间和市值均为构造教学输入；Apple 小节只核验其 SEC 财报行。Notebook 不下载行情或宣称真实估值、交易收益。

In [ ]:
from datetime import date, datetime, timezone
from pathlib import Path
import sys
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "quant_course" / "valuation.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Cannot locate project root containing src/quant_course/valuation.py")
sys.path.insert(0, str(ROOT / "src"))
from quant_course.valuation import (
    Amount, FinancialVintage, MarketCapSnapshot, constructed_company_cases,
    discounted_cash_flow_value, enterprise_multiple, enterprise_value,
    earnings_yield, equity_from_enterprise_value, fcf_yield, fcff_from_ebit,
    fcfe_from_net_income, gordon_terminal_value, industry_percentile_ranks,
    pit_fcf_yield, price_to_book, price_to_earnings,
    weighted_average_cost_of_capital,
)
print("项目目录：", ROOT)
print("单位规则：金额必须先匹配 currency 与 unit")

## 从股价与财报索取权得到倍数

以下两家企业是构造数据。计算链从构造价格和百万股数得到股权市值，再桥接到覆盖经营索取权的 EV。

In [ ]:
cases = constructed_company_cases()
results = {}
for name, case in cases.items():
    cap = case["market_cap"]
    ev = enterprise_value(
        cap, case["debt"], case["cash"], noncontrolling_interest=case["nci"],
        preferred_equity=case["preferred"], nonoperating_assets=case["nonoperating_assets"],
    )
    results[name] = {
        "EV": ev.value,
        "Equity from bridge": equity_from_enterprise_value(
            ev, case["debt"], case["cash"], noncontrolling_interest=case["nci"],
            preferred_equity=case["preferred"], nonoperating_assets=case["nonoperating_assets"],
        ).value,
        "P/E": price_to_earnings(cap, case["net_income"]),
        "P/B": price_to_book(cap, case["book_equity"]),
        "EV/EBITDA": enterprise_multiple(ev, case["ebitda"]),
        "EV/EBIT": enterprise_multiple(ev, case["ebit"]),
        "Earnings yield": earnings_yield(cap, case["net_income"]),
        "FCFE/equity": fcf_yield(case["fcfe"], cap, cash_flow_basis="FCFE", claim_basis="Equity"),
        "FCFF/EV": fcf_yield(case["fcff"], ev, cash_flow_basis="FCFF", claim_basis="EV"),
    }
assert results["Orion"]["EV"] == 650
assert results["Lyra"]["EV"] == 750
assert results["Orion"]["P/E"] == 20
assert results["Lyra"]["P/E"] == 24
results

Orion 与 Lyra 的 EV/EBITDA 分别约 8.67x、10x。两家 EBITDA 相同，EV 不同，因为资本索取权、现金和非经营资产不同。FCFE/市值和 FCFF/EV 各自使用了匹配分母。

In [ ]:
display(Image(filename=str(ROOT / "figures" / "a3" / "ch13_ev_bridge.png")))

## 从会计现金流到 DCF

FCFF 从税后经营利润起步；FCFE 从已经扣融资后的普通股净利润起步。下列构造金额都用 USD millions。

In [ ]:
fcff = fcff_from_ebit(
    Amount(50, "USD", "USD millions"), 0.25,
    Amount(10, "USD", "USD millions"), Amount(15, "USD", "USD millions"),
    Amount(5, "USD", "USD millions"),
)
fcfe = fcfe_from_net_income(
    Amount(30, "USD", "USD millions"), Amount(10, "USD", "USD millions"),
    Amount(15, "USD", "USD millions"), Amount(5, "USD", "USD millions"),
    Amount(5, "USD", "USD millions"),
)
orion = cases["Orion"]
wacc = weighted_average_cost_of_capital(
    orion["market_cap"], orion["debt"], cost_of_equity=0.10,
    cost_of_debt=0.06, tax_rate=0.25,
)
forecast = [Amount(x, "USD", "USD millions") for x in (40, 45, 50, 55, 60)]
base_wacc, stable_growth = 0.09, 0.03
terminal = gordon_terminal_value(forecast[-1], base_wacc, stable_growth)
dcf_ev = discounted_cash_flow_value(
    forecast, base_wacc, cash_flow_basis="FCFF", discount_rate_basis="WACC",
    terminal_value=terminal,
)
pv_terminal = terminal.value / (1 + base_wacc) ** len(forecast)
assert fcff.value == 27.5 and fcfe.value == 25
assert round(terminal.value, 2) == 1030.00
assert round(dcf_ev.value, 2) == 860.57
assert round(pv_terminal / dcf_ev.value * 100, 2) == 77.79
{
    "constructed FCFF": fcff.value,
    "constructed FCFE": fcfe.value,
    "Orion constructed WACC": wacc,
    "terminal value at year 5": terminal.value,
    "DCF enterprise value": dcf_ev.value,
    "PV terminal share": pv_terminal / dcf_ev.value,
}

In [ ]:
display(Image(filename=str(ROOT / "figures" / "a3" / "ch13_terminal_sensitivity.png")))

冻结敏感性图中的空格对应增长率g≥WACC，已超出增长永续现金流求和的收敛域。g=WACC时分母为零；g>WACC时代数商仍可计算，但其所代表的无限贴现现金流级数发散，不能用作该永续价值。终值占比高说明估值敏感，不代表模型可信。这里显示已有教学图，不重新生成图。

## PIT 版本、市场时点与行业截面

财报结束日不等于可用日。以下时间、版本号和价格均为构造值。信号截点在 2025-03-31，只能使用已经公开且入库的原始 FY2024 版本；下一版市值和之后重述不得回填。

In [ ]:
utc = timezone.utc
versions = [
    FinancialVintage(
        issuer_id="SIM-ORION", security_id="SIM-ORION-COMMON", fiscal_period="FY2024",
        period_end=date(2024, 12, 31), available_at=datetime(2025, 2, 15, 8, tzinfo=utc),
        ingested_at=datetime(2025, 2, 16, 10, tzinfo=utc), accession="SIM-V1",
        currency="USD", unit="USD millions", net_income_common=30, common_book_equity=180,
        ebit=50, ebitda=75, free_cash_flow=28, cash_flow_basis="FCFE", industry="Software",
    ),
    FinancialVintage(
        issuer_id="SIM-ORION", security_id="SIM-ORION-COMMON", fiscal_period="FY2024",
        period_end=date(2024, 12, 31), available_at=datetime(2025, 4, 2, 8, tzinfo=utc),
        ingested_at=datetime(2025, 4, 2, 9, tzinfo=utc), accession="SIM-V2",
        currency="USD", unit="USD millions", net_income_common=27, common_book_equity=177,
        ebit=48, ebitda=73, free_cash_flow=25, cash_flow_basis="FCFE", industry="Software",
    ),
]
signal_time = datetime(2025, 3, 31, 20, tzinfo=utc)
caps = [
    MarketCapSnapshot("SIM-ORION-COMMON", signal_time, Amount(600, "USD", "USD millions")),
    MarketCapSnapshot("SIM-ORION-COMMON", datetime(2025, 4, 2, 20, tzinfo=utc),
                      Amount(620, "USD", "USD millions")),
]
value, chosen_version, chosen_cap = pit_fcf_yield(
    versions, caps, security_id="SIM-ORION-COMMON", fiscal_period="FY2024", signal_time=signal_time,
)
assert chosen_version.accession == "SIM-V1"
assert chosen_cap.market_cap.value == 600
assert value == 28 / 600
print({"signal_time": signal_time.isoformat(), "accession": chosen_version.accession,
       "market_cap_as_of": chosen_cap.as_of.isoformat(), "FCFE yield": value})

In [ ]:
late_signal = datetime(2025, 4, 3, 20, tzinfo=utc)
late_value, late_version, late_cap = pit_fcf_yield(
    versions, caps, security_id="SIM-ORION-COMMON", fiscal_period="FY2024", signal_time=late_signal,
)
assert late_version.accession == "SIM-V2"
assert late_cap.market_cap.value == 620
assert late_value == 25 / 620
values = {"A": 0.05, "B": 0.03, "C": 0.03, "LOSS": None, "BANK": 0.08}
industry = {"A": "Software", "B": "Software", "C": "Software",
            "LOSS": "Software", "BANK": "Banks"}
ranks = industry_percentile_ranks(values, industry)
assert ranks["A"] == 1.0 and ranks["B"] == ranks["C"] == 0.25
assert ranks["LOSS"] is None and ranks["BANK"] == 0.5
ranks

年度财报频率和信号截点明确。教学 signal 在 2025-03-31 20:00 UTC，构造 trade_time 可放在其后；若构造次日成交，收益区间只能从成交生效后起算。本 Notebook 不计算回报或任何因子绩效。

## Apple SEC 申报核验

Apple FY2024 10-K 报告期间截至 2024-09-28，单位说明为金额 USD millions、股数 thousands。SEC 原申报列净利润 93,736、摊薄 EPS USD 6.08、加权平均摊薄股数 15,408,095 thousand shares。以下仅复核财报算术，不把没有验证的股价补成真实 P/E。来源：[Apple FY2024 Form 10-K](https://www.sec.gov/Archives/edgar/data/320193/000032019324000123/aapl-20240928.htm)。

In [ ]:
net_income_millions = 93_736
weighted_average_diluted_shares_thousands = 15_408_095
calculated_diluted_eps = net_income_millions * 1_000 / weighted_average_diluted_shares_thousands
reported_diluted_eps = 6.08
assert round(calculated_diluted_eps, 2) == reported_diluted_eps
print(f"SEC-stated EPS={reported_diluted_eps:.2f}; arithmetic from rounded filing rows={calculated_diluted_eps:.4f}")
print("No market price is loaded, so this is not a P/E calculation.")